In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch.utils.data import Subset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 10

In [ ]:
#weights = models.ResNet18_Weights.DEFAULT
#model = models.resnet18(weights=weights) #pretrained
#preprocess = weights.transforms()

model = models.resnet18(weights=None) #treniranje iz nule


model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device) 


for p in model.parameters():
    p.requires_grad = True

optimiser = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(), #ovdje nadodavati augmentacije
])

val_transform = transforms.Compose([
    transforms.ToTensor(), #nadodati std i mean
])

base_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=None
)

N = len(base_dataset)
perm_img = torch.randperm(N, generator=torch.Generator().manual_seed(42))
train_index = perm_img[:int(0.8*N)].tolist()
val_index = perm_img[int(0.8*N):].tolist()

train_dataset = datasets.CIFAR10(
    root="./data", 
    train=True, 
    download=False, 
    transform=train_transform)

val_dataset = datasets.CIFAR10(
    root="./data", 
    train=True, 
    download=False, 
    transform=val_transform)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

#DONIK: Interesantno, mozete pogledati i nacin podjele podataka s SubsetRandomSampler koji se koristi direktno u DataLoaderu

train_dataset = Subset(train_dataset, train_index)
val_dataset   = Subset(val_dataset, val_index)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)


In [ ]:
def evaluate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            prediction = outputs.argmax(dim=1)
            correct += (prediction == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss / total
    val_acc = correct / total
    return val_loss, val_acc

In [ ]:
epohs = 200

for i in range(epohs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item() * images.size(0)
        prediction = outputs.argmax(dim=1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    print(f"Epoch {i+1}/{epohs} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f}")

    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {i+1}/{epohs} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

In [ ]:
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)
        prediction = outputs.argmax(dim=1)
        correct += (prediction == labels).sum().item()
        total += labels.size(0)

test_loss = test_loss / total
test_acc = correct / total
print(f"test_loss={test_loss:.4f} | test_acc={test_acc:.4f}")